<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 13. GAN — Generative Adversarial Nets

- **원 논문:** [Generative Adversarial Nets](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html)
- **저자 / 발표:** Ian J. Goodfellow et al. · NeurIPS (2014)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** PyTorch 기본, binary cross entropy, 확률분포, 역전파

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

2차원 Gaussian 데이터에 대해 MLP 생성자와 판별자를 번갈아 갱신하고, 논문이 제안한 non-saturating 생성자 목적함수로 고정 noise 표본의 분포 오차가 줄어드는지 확인한다.

**원 논문과 다른 것**

논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3, Eq. (1) | `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수 | 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사 |
| §3, Algorithm 1 및 Eq. (1) 직후 문단 | `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신 | 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사 |
| §4.1, Proposition 1 Eq. (2), Theorem 1 및 Eq. (6) | `rbf_mmd`와 판별 확률로 생성 분포 수렴을 진단 | 학습 전후 MMD와 판별자 평균 출력을 비교 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\min_G\max_D\;
\mathbb{E}_{x\sim p_{data}}[\log D(x)]
+\mathbb{E}_{z\sim p(z)}[\log(1-D(G(z)))]
$$

- generator $G$는 noise를 sample로, discriminator $D$는 sample을 real 확률로 변환합니다.
- 두 모델의 목표가 반대인 minimax game이므로 optimizer와 gradient 경로를 분리합니다.
- discriminator step과 generator step을 독립 함수로 구현하고 해당 step의 parameter만 변하는지 보장합니다.
- $z:[B,D_z]\rightarrow G(z):[B,D_x]\rightarrow D(\cdot):[B,1]$의 흐름입니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3, Eq. (1)
- **노트북 구현:** `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문의 게임을 2차원으로 축소하기

논문 §3 Eq. (1)은

$$\min_G\max_D\;\mathbb E_{x\sim p_{data}}\log D(x)
+\mathbb E_{z\sim p_z}\log(1-D(G(z)))$$

를 정의합니다. 여기서는 타원형 2차원 Gaussian을 실제 분포로 두고 두 MLP를
번갈아 갱신합니다. 생성자에는 같은 절에서 권한 `maximize log D(G(z))`, 즉
non-saturating loss를 사용합니다. 이 변형은 고정점은 같고 초기 gradient가 더 강합니다.

**성공 기준:** 고정 noise에서 생성 분포와 실제 분포 사이 RBF-MMD가 초기보다 작아야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,np.random.seed,torch.tensor,torch.randn,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(130)
np.random.seed(130)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# This tiny teaching batch may be faster with AI_LAB_DEVICE=cpu;
# auto acceleration preserves the same code path for larger variants.

real_mean = torch.tensor([1.4, -0.8])
real_transform = torch.tensor([[0.75, 0.0], [0.28, 0.42]])


def sample_real(n, generator):
    base = torch.randn(n, 2, generator=generator)
    return (real_mean + base @ real_transform.T).to(DEVICE)


def sample_noise(n, generator):
    return torch.randn(n, 2, generator=generator).to(DEVICE)


data_rng = torch.Generator().manual_seed(131)
real_data = sample_real(1024, data_rng)
print(ACCELERATOR.summary())
print(f"real mean={real_data.mean(0).tolist()}, shape={tuple(real_data.shape)}")
assert real_data.shape == (1024, 2)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3, Eq. (1)
- **이 셀의 책임:** `discriminator_loss`와 `generator_loss`의 두 플레이어 목적함수
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 실제·가짜 logit별 BCE와 양쪽 parameter gradient를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.Linear,F.binary_cross_entropy_with_logits,ACCELERATOR.move,torch.zeros -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>F.binary_cross_entropy_with_logits(input, target, ...)</code></summary>

#### `F.binary_cross_entropy_with_logits(input, target, ...)`

- **역할:** sigmoid와 binary cross entropy를 안정적으로 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{BCE}=-\frac{1}{N}\sum_i\left[y_i\log\sigma(z_i)+(1-y_i)\log(1-\sigma(z_i))\right]
$$

- **기호:** $z_i$는 logit, $y_i\in[0,1]$은 target, $\sigma$는 sigmoid입니다.
- **수식 → 코드:** sigmoid와 BCE를 한 함수에서 log-sum-exp 형태로 안정적으로 계산합니다.
- **직관:** 각 항을 독립적인 이진 선택으로 보고 정답 쪽 확률의 음의 로그를 최소화합니다.
- **shape 확인:** logit과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.binary_cross_entropy_with_logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.binary_cross_entropy_with_logits.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 2),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def discriminator_loss(real_logits, fake_logits):
    real_term = F.binary_cross_entropy_with_logits(
        real_logits,
        torch.ones_like(real_logits),
    )
    fake_term = F.binary_cross_entropy_with_logits(
        fake_logits,
        torch.zeros_like(fake_logits),
    )
    return real_term + fake_term


def generator_loss(fake_logits):
    # §3: minimax와 고정점이 같지만 초기 포화를 줄이는 대안.
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


G, D = ACCELERATOR.move(Generator(), Discriminator())
assert G(torch.zeros(5, 2, device=DEVICE)).shape == (5, 2)
assert D(torch.zeros(5, 2, device=DEVICE)).shape == (5,)
zero_logits = torch.zeros(2, device=DEVICE)
assert float(discriminator_loss(zero_logits, zero_logits)) > 0.0

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3, Algorithm 1 및 Eq. (1) 직후 문단
- **이 셀의 책임:** `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Optimizer.zero_grad,Tensor.detach,Tensor.backward,Optimizer.step,Module.parameters,Tensor.requires_grad_ -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.requires_grad_(requires_grad=True)</code></summary>

#### `tensor.requires_grad_(requires_grad=True)`

- **역할:** 텐서의 gradient 추적 여부를 in-place로 바꿉니다.
- **입력·반환:** 불리언을 받고 같은 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** teacher 동결이나 새로 학습할 파라미터 범위를 명시합니다.
- **주의점:** 이미 만들어진 계산 그래프를 소급해 바꾸지 않으며 leaf 여부를 확인해야 합니다.
- **공식 문서:** [Tensor.requires_grad_](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.requires_grad_.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def gan_step(G, D, g_optimizer, d_optimizer, batch_size, generator):
    # Algorithm 1: k=1 discriminator update.
    real_batch = sample_real(batch_size, generator)
    noise = sample_noise(batch_size, generator)
    fake_batch = G(noise)
    d_optimizer.zero_grad()
    d_loss = discriminator_loss(D(real_batch), D(fake_batch.detach()))
    d_loss.backward()
    d_optimizer.step()

    # §3의 non-saturating 대안으로 generator update.
    noise = sample_noise(batch_size, generator)
    g_optimizer.zero_grad()
    for parameter in D.parameters():
        parameter.requires_grad_(False)
    g_loss = generator_loss(D(G(noise)))
    g_loss.backward()
    g_optimizer.step()
    for parameter in D.parameters():
        parameter.requires_grad_(True)
    return float(d_loss.detach()), float(g_loss.detach())


# detach가 없으면 이 검사 시 D update가 G에 불필요한 gradient를 남길 수 있다.

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3, Algorithm 1 및 Eq. (1) 직후 문단
- **이 셀의 책임:** `gan_step`의 판별자 1회 갱신 뒤 생성자 1회 갱신
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 optimizer의 gradient 경로가 분리되고 loss가 유한한지 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.no_grad,torch.optim.Adam,np.isfinite,np.asarray -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

<details>
<summary><code>np.asarray(a, dtype=None, order=None)</code></summary>

#### `np.asarray(a, dtype=None, order=None)`

- **역할:** 가능하면 복사 없이 입력을 NumPy 배열로 해석합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.asarray](https://numpy.org/doc/stable/reference/generated/numpy.asarray.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
train_rng = torch.Generator().manual_seed(132)
probe_rng = torch.Generator().manual_seed(133)
probe_noise = sample_noise(384, probe_rng)
with torch.no_grad():
    initial_fake = G(probe_noise).clone()

g_optimizer = torch.optim.Adam(G.parameters(), lr=1.5e-3, betas=(0.5, 0.9))
d_optimizer = torch.optim.Adam(D.parameters(), lr=1.0e-3, betas=(0.5, 0.9))
history = []
for step in range(600):
    history.append(gan_step(G, D, g_optimizer, d_optimizer, 128, train_rng))

assert len(history) == 600
assert np.isfinite(np.asarray(history)).all()
print(f"final D loss={history[-1][0]:.3f}, G loss={history[-1][1]:.3f}")

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4.1, Proposition 1 Eq. (2), Theorem 1 및 Eq. (6)
- **이 셀의 책임:** `rbf_mmd`와 판별 확률로 생성 분포 수렴을 진단
- **Tensor shape 계약:** noise [B,D_z] → samples [B,Dₓ] → discriminator logits [B,1]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 학습 전후 MMD와 판별자 평균 출력을 비교. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.cdist,torch.exp,torch.sigmoid -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.cdist(x1, x2, p=2.0, compute_mode=...)</code></summary>

#### `torch.cdist(x1, x2, p=2.0, compute_mode=...)`

- **역할:** 두 점 집합의 모든 쌍별 p-norm 거리를 계산합니다.
- **입력·반환:** 두 batch 점 집합을 받고 pairwise distance 행렬을 반환합니다.
- **이 셀에서 쓰는 이유:** clustering, nearest neighbor, representation separation을 벡터화합니다.
- **주의점:** 두 입력의 feature 차원과 batch broadcasting 조건이 같아야 합니다.

##### 관련 수식과 코드 연결

$$
D_{ij}=\lVert x_i-y_j\rVert_p=\left(\sum_{k=1}^{D}\lvert x_{ik}-y_{jk}\rvert^p\right)^{1/p}
$$

- **기호:** $i,j$는 두 점 집합의 index이고 $k$는 feature 축입니다.
- **수식 → 코드:** API가 모든 $i,j$ 조합의 거리를 Python loop 없이 계산합니다.
- **직관:** 가까운 표현은 작은 값, 멀리 떨어진 표현은 큰 값으로 나타납니다.
- **shape 확인:** `[B,P,D]`와 `[B,R,D]`를 받으면 `[B,P,R]`을 반환합니다.
- **공식 문서:** [torch.cdist](https://docs.pytorch.org/docs/stable/generated/torch.cdist.html)

</details>

<details>
<summary><code>torch.exp(input)</code></summary>

#### `torch.exp(input)`

- **역할:** 각 원소에 자연지수함수 $e^x$를 적용합니다.
- **입력·반환:** 실수 텐서를 받고 같은 shape의 양수 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit, 확률비, diffusion coefficient를 양수 scale로 바꿉니다.
- **주의점:** 큰 양수는 overflow, 큰 음수는 0에 가까운 underflow를 만들 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=e^{x_i}
$$

- **기호:** $e$는 자연로그의 밑입니다.
- **수식 → 코드:** API가 각 원소를 양수 값으로 지수 변환합니다.
- **직관:** 더하기 차이를 곱셈 비율로 바꾸며 log-domain 값을 원래 scale로 복원합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.exp](https://docs.pytorch.org/docs/stable/generated/torch.exp.html)

</details>

<details>
<summary><code>torch.sigmoid(input)</code></summary>

#### `torch.sigmoid(input)`

- **역할:** 실수를 0과 1 사이 값으로 바꿉니다.
- **입력·반환:** logit 텐서를 받고 같은 shape의 확률형 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 이진 확률, gate, attention gate를 표현합니다.
- **주의점:** 큰 절댓값에서는 gradient가 거의 0이 되며 BCE 학습은 logits 버전이 안정적입니다.

##### 관련 수식과 코드 연결

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

- **기호:** $x$는 제한되지 않은 logit이고 $\sigma(x)$는 0과 1 사이입니다.
- **수식 → 코드:** 코드가 logit을 이진 확률이나 gate 값으로 변환합니다.
- **직관:** 0 근처는 민감하고 큰 절댓값에서는 0 또는 1에 포화됩니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sigmoid](https://docs.pytorch.org/docs/stable/generated/torch.sigmoid.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def rbf_mmd(x, y, sigma=1.0):
    def kernel(a, b):
        squared_distance = torch.cdist(a, b).pow(2)
        return torch.exp(-squared_distance / (2.0 * sigma**2))

    return kernel(x, x).mean() + kernel(y, y).mean() - 2.0 * kernel(x, y).mean()


eval_rng = torch.Generator().manual_seed(134)
real_eval = sample_real(len(probe_noise), eval_rng)
with torch.no_grad():
    final_fake = G(probe_noise)
    initial_mmd = float(rbf_mmd(initial_fake, real_eval))
    final_mmd = float(rbf_mmd(final_fake, real_eval))
    mean_error = float((final_fake.mean(0) - real_eval.mean(0)).norm())
    d_real = float(torch.sigmoid(D(real_eval)).mean())
    d_fake = float(torch.sigmoid(D(final_fake)).mean())

print(f"RBF-MMD: {initial_mmd:.4f} -> {final_mmd:.4f}")
print(f"mean error={mean_error:.3f}, D(real)={d_real:.3f}, D(fake)={d_fake:.3f}")
assert math.isfinite(final_mmd) and final_mmd < initial_mmd
assert mean_error < 0.45

losses = np.asarray(history)
real_eval_cpu = real_eval.detach().cpu()
initial_fake_cpu = initial_fake.detach().cpu()
final_fake_cpu = final_fake.detach().cpu()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].scatter(
    real_eval_cpu[:, 0],
    real_eval_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="real",
)
axes[0].scatter(
    initial_fake_cpu[:, 0],
    initial_fake_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="initial G",
)
axes[0].set_title("before training")
axes[0].legend()
axes[1].scatter(
    real_eval_cpu[:, 0],
    real_eval_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="real",
)
axes[1].scatter(
    final_fake_cpu[:, 0],
    final_fake_cpu[:, 1],
    s=10,
    alpha=0.55,
    label="final G",
)
axes[1].set_title("after adversarial training")
axes[1].legend()
axes[2].plot(losses[:, 0], alpha=0.75, label="D loss")
axes[2].plot(losses[:, 1], alpha=0.75, label="G loss")
axes[2].set(title="Algorithm 1 updates", xlabel="step")
axes[2].legend()
fig.tight_layout()
plt.show()

## 7. 해석 예시 답안

1. 두 목적함수가 함께 변하는 게임이므로 한 모델이 좋아지면 상대 모델의 loss가
   다시 커질 수 있습니다. 따라서 표본과 분포 metric도 함께 봐야 합니다.
2. 판별자 단계에서 D의 loss가 G까지 역전파되어 G를 갱신하는 경로를 끊습니다. G는 별도의 생성자 단계에서만 갱신됩니다.
3. $p_g=p_{data}$이면 최적 판별자 $D^*(x)=1/2$이므로 두 평균 모두 0.5에
   가까워집니다. 유한 MLP·유한 step에서는 정확히 0.5일 필요는 없습니다.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `논문은 MNIST, TFD, CIFAR-10에서 MLP GAN을 학습한다. 여기서는 2차원 점 1,024개와 작은 MLP를 사용해 Algorithm 1의 경쟁적 업데이트만 수 초 내에 재현한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.